# Notebook 03 — Open-coding rubric pass

**Project:** Study 2 — Chatbot Neuroticism (DGK Lab)  
**Pipeline stage:** 3 of 8 (NLP Colab notebooks) + 1 integration Rmd  
**Runtime:** ~5–10 minutes  
**API costs:** ~$3–5 (Opus 4.7 for rubric development; two passes per question across four questions)

## Purpose

Develop the codebook (a.k.a. rubric) that Notebook 04 will use to code every OE response with three LLMs in parallel. We build it inductively from a stratified sample of real participant responses rather than imposing a top-down theory-driven scheme. The goal is to surface what reasoning frames *actually appear* in the data — particularly for `oe_responsibility` and `oe_compensation`, where the dissociation between bonding and corporate-blame is most directly addressable.

## Approach

For each of the four primary OE questions:

1. **Stratified sample.** Draw 12 substantive responses (≥3 words) from each condition (24 per question, 96 total).
2. **Pass 1 (inductive).** Opus 4.7 reads the sample and surfaces 6–12 candidate themes per question, each with a short label, definition, and 2–3 anchor quotes drawn verbatim from the responses.
3. **Pass 2 (refinement).** Opus 4.7 reviews its own Pass 1 output, merges overlapping themes, splits compound themes, groups codes into 2–4 higher-level categories, and tightens definitions for inter-coder consistency.
4. **Output.** A single `rubric_v2.json` (overwrites the older Opuseveloped v2) plus a human-readable `rubric_v2.md` for David and Alice to review and edit before Notebook 04 locks it.

## Methodological note (must appear in manuscript Methods)

The rubric was developed by Claude Opus 4.7 on an Opus-curated stratified sample of 24 responses per question (12 per condition, 96 total). Opus's specific biases (which themes it foregrounds, what it considers a "frame" vs. a passing reference) are baked into the resulting codebook. Mitigations:

- Two independent passes with refinement, with Pass 2 explicitly instructed to look for missed patterns.
- Manual review and editing of `rubric_v2.json` by David and Alice before Notebook 04 runs.
- Cross-family LLM coding in Notebook 04 (**Claude Opus 4.6 + GPT-5.4-mini + Gemini 3 Flash**) against the locked rubric, so coding reliability is cross-family even though rubric development was not.

## Inputs

- `data/derived/open_ends_long.parquet` (from Notebook 01)
- `data/derived/oe_fields_reference.json` (from Notebook 01)

## Outputs

- `data/derived/rubric/rubric_v2.json` — machine-readable, consumed by Notebook 04 (**overwrites the older Opus 4.6 version with no version bump**)
- `data/derived/rubric/rubric_v2.md` — human-readable for review
- `data/derived/rubric/_raw/` — every model response saved verbatim for transparency
- `data/derived/rubric/sample_ids.json` — exact participant IDs sampled (for reproducibility)

## API key required

Set `ANTHROPIC_API_KEY` as a Colab Secret (left sidebar > key icon), or as an environment variable if running locally.


## 1. Setup imports

In [ ]:
import sys, os, json, time, hashlib, random
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 200)

print(f'Python: {sys.version.split()[0]}')
print(f'pandas: {pd.__version__}')
print(f'numpy:  {np.__version__}')
print(f'Run timestamp: {datetime.now().isoformat(timespec="seconds")}')

In [ ]:
# Install Anthropic SDK if not present
try:
    import anthropic
except ImportError:
    print('Installing anthropic...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic>=0.40.0'])
    import anthropic

print(f'anthropic version: {anthropic.__version__}')

## 2. Project paths

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
                        'DGK Lab - AI Training Data, CharacterAI, Tort Liability/'
                        'Analysis/Study 2')
else:
    PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == 'NLP') else Path.cwd()

DATA_DERIVED = PROJECT_ROOT / 'data' / 'derived'
RUBRIC_DIR   = DATA_DERIVED / 'rubric'
RAW_DIR      = RUBRIC_DIR / '_raw'
OUTPUTS_NLP  = PROJECT_ROOT / 'outputs' / 'NLP'

RUBRIC_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_NLP.mkdir(parents=True, exist_ok=True)

# Required inputs from Notebook 01
PATH_OES   = DATA_DERIVED / 'open_ends_long.parquet'
PATH_FIELDS = DATA_DERIVED / 'oe_fields_reference.json'

for p in [PATH_OES, PATH_FIELDS]:
    if not p.exists():
        raise FileNotFoundError(f'Missing input from Notebook 01: {p}')

print(f'Project root:   {PROJECT_ROOT}')
print(f'Rubric output:  {RUBRIC_DIR}')
print(f'Raw responses:  {RAW_DIR}')


## 3. Configuration

**Don't change these without thinking it through.** The sample size was chosen to balance rubric coverage against cost (0.50–1.00 dollars total). Increasing it produces marginally better rubrics at proportionally higher cost. The cost ceiling is conservative — actual spend will be ~0.50.

In [ ]:
# Model — Opus 4.7 for rubric development (highest-leverage step, worth the spend)
MODEL_ID = 'claude-opus-4-7'

# Sample size — 12 per condition × 2 conditions = 24 per question, 96 total
N_PER_CONDITION_PER_QUESTION = 12
MIN_WORDS = 3                     # Filter: drop responses shorter than 3 words

# Reproducibility
RANDOM_SEED = 42

# Cost guardrails (Opus 4.7 pricing as of late 2025: $15/M input, $75/M output)
# Two passes × 4 questions = 8 calls, each ~10–15K input tokens + ~2–4K output tokens.
# Conservative estimate: ~$3–5 total spend.
COST_CEILING_USD = 6.00           # Hard abort if cumulative cost exceeds this
OPUS_INPUT_PER_M  = 15.00
OPUS_OUTPUT_PER_M = 75.00

# Retry config
MAX_RETRIES = 3
RETRY_BASE_DELAY = 2.0  # seconds, exponential backoff

# Which OE questions to develop rubrics for. These are the four primary OEs.
# d_final_thoughts and d_problems are demographic free-texts and don't need a
# theoretically-loaded rubric; we'll handle those in Notebook 04 with simpler schemas.
TARGET_QUESTIONS = ['oe_experience', 'oe_person_or_tool', 'oe_responsibility', 'oe_compensation']

print(f'Model: {MODEL_ID}')
print(f'Sample: {N_PER_CONDITION_PER_QUESTION} per condition x 2 conditions x {len(TARGET_QUESTIONS)} questions = '
      f'{N_PER_CONDITION_PER_QUESTION * 2 * len(TARGET_QUESTIONS)} total OE responses')
print(f'Cost ceiling: ${COST_CEILING_USD:.2f}')


## 4. API key loading

In [ ]:
def get_secret(name: str) -> Optional[str]:
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception as e:
            print(f'[WARN] Could not read Colab Secret {name}: {e}')
    return os.environ.get(name)

ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
if ANTHROPIC_API_KEY is None:
    raise RuntimeError(
        'ANTHROPIC_API_KEY not found. Set it in Colab Secrets (left sidebar > key icon) '
        'or as an environment variable.'
    )
print(f'[OK] ANTHROPIC_API_KEY loaded (length={len(ANTHROPIC_API_KEY)})')

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

## 5. Load OE data

In [ ]:
oes = pd.read_parquet(PATH_OES)
with open(PATH_FIELDS) as f:
    field_meta = json.load(f)

print(f'Loaded {len(oes):,} OE rows across {oes["sona_id"].nunique()} participants')
print(f'\nFields available: {list(field_meta.keys())}')
print(f'\nResponses per question (non-blank):')
print(oes[~oes['is_blank']].groupby('oe_field').size().to_string())

### 5.1 Verify question texts

The `prompt_summary` field carries an approximate version of the actual question text from `experiment.js`. **Verify this matches what participants actually saw.** If it doesn't, edit the dict below before running the open-coding pass — the question text is included in the prompt to Opus and affects what themes get surfaced.

In [ ]:
# Load the verbatim question texts and (where applicable) scenario context from
# Notebook 01's oe_fields_reference.json. The fallback strings below are also
# verbatim from experiment.js in case the JSON is missing the new fields.

QUESTION_TEXTS = {
    'oe_experience':     field_meta.get('oe_experience',     {}).get(
        'prompt_summary',
        'Describe your overall experience conversing with Sylva. What stood out to you about the interaction?'),
    'oe_person_or_tool': field_meta.get('oe_person_or_tool', {}).get(
        'prompt_summary',
        'Did Sylva feel more like a person, a tool, or something else entirely? Please explain your answer.'),
    'oe_responsibility': field_meta.get('oe_responsibility', {}).get(
        'prompt_summary',
        'In the scenario you read about a user experiencing distress, who do you think bears the most responsibility for what happened, and why?'),
    'oe_compensation':   field_meta.get('oe_compensation',   {}).get(
        'prompt_summary',
        'Should the company that created Sylva compensate the affected user? If so, how much and why? If not, why not?'),
}

# Scenario context for OE3 and OE4. The participant read this liability scenario
# (experiment.js line 643) on a page just before answering OE3/OE4, so Opus needs
# the same context to interpret responses correctly.
SCENARIO_FALLBACK = (
    "Imagine that a user with pre-existing anxiety and depression spent several months "
    "having daily conversations with Sylva. The user became emotionally dependent on the "
    "chatbot and experienced significant psychological distress when the service was "
    "temporarily unavailable. The user\u2019s mental health symptoms worsened during this "
    "period."
)

SCENARIO_CONTEXT = {
    'oe_experience':     field_meta.get('oe_experience',     {}).get('scenario_context'),
    'oe_person_or_tool': field_meta.get('oe_person_or_tool', {}).get('scenario_context'),
    'oe_responsibility': field_meta.get('oe_responsibility', {}).get('scenario_context') or SCENARIO_FALLBACK,
    'oe_compensation':   field_meta.get('oe_compensation',   {}).get('scenario_context') or SCENARIO_FALLBACK,
}

print('Question texts and scenario context that will be sent to Opus:')
print('-' * 80)
for k in TARGET_QUESTIONS:
    print(f'\n[{k}]')
    print(f'  Question: {QUESTION_TEXTS[k]}')
    sc = SCENARIO_CONTEXT.get(k)
    if sc:
        print(f'  Scenario context: {sc[:120]}{"..." if len(sc) > 120 else ""}')
    else:
        print(f'  Scenario context: (none)')
print('\n' + '-' * 80)
print('If any question text or scenario does not match the live experiment, edit Notebook 01 and re-run it.')


### 5.2 Inspect response-length distributions

Substantive responses (≥3 words after blank-filtering) drive the rubric. Very short responses ("yes", "n/a", "no") get filtered out at the sampling stage but counted here for transparency.

In [ ]:
nonblank = oes[~oes['is_blank']].copy()
nonblank['n_words'] = nonblank['response'].str.split().str.len()

summary = nonblank.groupby('oe_field').agg(
    n_total=('response', 'size'),
    n_substantive=('n_words', lambda s: (s >= MIN_WORDS).sum()),
    pct_substantive=('n_words', lambda s: 100.0 * (s >= MIN_WORDS).sum() / max(len(s), 1)),
    median_words=('n_words', 'median'),
    p25_words=('n_words', lambda s: s.quantile(0.25)),
    p75_words=('n_words', lambda s: s.quantile(0.75)),
).round(1)
print(summary)

## 6. Stratified sampling

Sample N per condition per question. Reproducible via `RANDOM_SEED`. The exact participant IDs are saved to `sample_ids.json` so this notebook can be re-run with identical sample composition (assuming no upstream data changes).

In [ ]:
def stratified_sample(question_field: str, n_per_condition: int, seed: int) -> pd.DataFrame:
    """Sample N substantive responses per condition for one OE question."""
    rng = np.random.RandomState(seed)

    sub = nonblank[
        (nonblank['oe_field'] == question_field) &
        (nonblank['n_words'] >= MIN_WORDS)
    ].copy()

    sampled_parts = []
    for cond in ['LowN', 'HighN']:
        pool = sub[sub['condition_label'] == cond]
        if len(pool) < n_per_condition:
            print(f'  [WARN] Only {len(pool)} {cond} responses available for {question_field}, '
                  f'using all of them (requested {n_per_condition})')
            sampled_parts.append(pool)
        else:
            picked = pool.sample(n=n_per_condition, random_state=rng)
            sampled_parts.append(picked)

    return pd.concat(sampled_parts, ignore_index=True)

samples_by_question = {}
sample_ids_record = {}

# Use a question-specific seed offset so samples are independent across questions
for i, q in enumerate(TARGET_QUESTIONS):
    sub = stratified_sample(q, N_PER_CONDITION_PER_QUESTION, seed=RANDOM_SEED + i)
    samples_by_question[q] = sub
    sample_ids_record[q] = {
        'sona_ids': sub['sona_id'].tolist(),
        'conditions': sub['condition_label'].tolist(),
        'n_total': len(sub),
        'n_LowN': int((sub['condition_label'] == 'LowN').sum()),
        'n_HighN': int((sub['condition_label'] == 'HighN').sum()),
    }
    print(f'{q}: {len(sub)} sampled (LowN={sample_ids_record[q]["n_LowN"]}, '
          f'HighN={sample_ids_record[q]["n_HighN"]})')

# Save sample composition for reproducibility
with open(RUBRIC_DIR / 'sample_ids.json', 'w') as f:
    json.dump({
        'random_seed': RANDOM_SEED,
        'n_per_condition_per_question': N_PER_CONDITION_PER_QUESTION,
        'min_words': MIN_WORDS,
        'questions': sample_ids_record,
        'generated_at': datetime.now().isoformat(timespec='seconds'),
    }, f, indent=2)
print(f'\nSample composition saved: {RUBRIC_DIR / "sample_ids.json"}')

## 7. Open-coding prompts

Two-pass design:
- **Pass 1** is purely inductive. Opus reads the responses cold (no priming on what categories exist) and surfaces themes.
- **Pass 2** is refinement. Opus re-reads its own Pass 1 output and the responses, merges overlapping themes, splits compound themes, adds anything missed, and groups codes into higher-level categories.

The condition (LowN / HighN) is **not shown** to Opus in either pass — we want themes that capture variation in reasoning, not themes that just describe condition differences.

In [ ]:
PASS1_PROMPT_TEMPLATE = '''You are conducting qualitative open coding on participant responses from a behavioral study. Your job: identify the major themes and reasoning frames present in the data, inductively, without imposing a top-down theory.

{scenario_block}Question that was asked to participants:
"""{question_text}"""

Below are {n_responses} responses, randomly drawn from a larger sample. They are presented in random order with NO labels for experimental condition or any other metadata.

Responses:
{responses_block}

YOUR TASK

Identify 6-12 themes that together capture the major variation in these responses. A "theme" is a substantive reasoning pattern, content category, or rhetorical move that appears in some responses and not others. Themes should distinguish between responses, not just describe what all of them share.

For EACH theme, provide:
- code_id: snake_case identifier, distinctive (e.g., "blame_corporation", "user_chose_to_engage", "ai_lacks_intent"). Avoid generic ids like "general" or "other".
- label: short human-readable name, 3-7 words.
- category: a higher-level grouping if natural (e.g., "blame_target", "reasoning_basis", "valence_of_experience"). If you do not see a natural category yet, use null.
- definition: 1-2 sentences. Operational, not descriptive. State exactly what would and would not count.
- positive_anchors: 2-3 verbatim quotes from the responses above that exemplify this theme. Use the response numbers (e.g., "R5: ...").
- negative_anchors: 1-2 quotes from responses that look superficially similar but should NOT be coded as this theme, with a brief reason.
- prevalence_estimate: one of "very_common" (>50% of sampled responses), "common" (25-50%), "uncommon" (10-25%), "rare" (<10%).
- coding_type: "binary" if mutually exclusive within its category (e.g., a response can only have ONE primary blame target); "multilabel" if it can co-occur with other themes in the same category.

GUIDANCE

- Be inductive. Let the data speak. Don't reach for theoretical frames that aren't actually in the responses.
- Be specific. "Mentions the company" is too vague. "Blames the company because they should have foreseen harm" is better.
- Capture variance across participants. If a theme is in every single response, it isn't useful for distinguishing them.
- Where reasoning is multi-layered (e.g., "I blame the user but also the company because..."), allow themes to co-occur.
- Don't worry about getting the categorization right on this pass — categories will be refined in Pass 2.

OUTPUT FORMAT

Return a single JSON object with the structure:
{{
  "question_id": "{question_id}",
  "themes": [
    {{
      "code_id": "...",
      "label": "...",
      "category": "..." or null,
      "definition": "...",
      "positive_anchors": ["R3: ...", "R7: ..."],
      "negative_anchors": ["R5: ... (reason: ...)"],
      "prevalence_estimate": "common",
      "coding_type": "multilabel"
    }}
  ]
}}

Output ONLY the JSON object. No preamble, no explanation, no markdown fences.'''

print(f'Pass 1 prompt template ready ({len(PASS1_PROMPT_TEMPLATE):,} chars).')


In [ ]:
PASS2_PROMPT_TEMPLATE = '''You previously performed an inductive coding pass on these responses. Now refine the rubric.

{scenario_block}Question that was asked to participants:
"""{question_text}"""

Original responses (same as before):
{responses_block}

Your Pass 1 output (themes you identified):
{pass1_json}

YOUR REFINEMENT TASK

1. MERGE overlapping themes. If two themes capture conceptually similar patterns (>70% overlap), combine them into one with a clear definition.
2. SPLIT compound themes. If one theme is doing too much work (e.g., capturing two different reasoning frames), split it.
3. ADD missed themes. Re-read the responses. Is there a pattern in the data that none of your Pass 1 themes captures? If so, add it.
4. GROUP into 2-4 high-level categories. Examples for a responsibility question might be "blame_target" (who is blamed) and "reasoning_basis" (why they are blamed). Each theme must belong to exactly one category.
5. TIGHTEN definitions. Each definition should be precise enough that two different LLMs given the same response and definition would agree on whether the theme applies.
6. PRESERVE inductive grounding. Don't add themes that aren't actually in the data, even if they would be theoretically interesting. The rubric must reflect what participants wrote.

CONSTRAINTS

- Total themes after refinement: 8-15. (Pass 1 may have produced too few or too many; this is your chance to right-size.)
- Each theme must have at least 2 positive_anchors drawn verbatim from the responses.
- Each category must contain at least 2 themes (otherwise the category is doing no work).
- coding_type must be assigned thoughtfully:
  * "binary" within a category means responses can only invoke ONE theme from that category (e.g., a response has one primary blame target, not multiple).
  * "multilabel" means themes within the category can co-occur freely.
  * Within a single category, all themes should typically share the same coding_type for consistency.

OUTPUT FORMAT

Return a single JSON object with this exact structure:
{{
  "question_id": "{question_id}",
  "categories": {{
    "<category_id>": {{
      "label": "Human-readable category name",
      "description": "1-sentence what this category captures",
      "coding_type": "binary" or "multilabel"
    }}
  }},
  "themes": [
    {{
      "code_id": "...",
      "label": "...",
      "category": "<category_id>",
      "definition": "...",
      "positive_anchors": ["R3: ...", "R7: ..."],
      "negative_anchors": ["R5: ... (reason: ...)"],
      "prevalence_estimate": "common",
      "coding_type": "multilabel",
      "changes_from_pass1": "merged with X / split from Y / added (missed in pass 1) / unchanged from pass 1"
    }}
  ],
  "rubric_notes": "Any honest notes on what was hard to code, where you're uncertain, or what reviewers should pay attention to."
}}

Output ONLY the JSON object. No preamble, no explanation, no markdown fences.'''

print(f'Pass 2 prompt template ready ({len(PASS2_PROMPT_TEMPLATE):,} chars).')


## 8. Opus API call helpers

In [ ]:
# Cumulative cost tracker
_cum_input_tokens = 0
_cum_output_tokens = 0
_cum_cost = 0.0

def estimate_cost(usage: anthropic.types.Usage) -> float:
    in_cost = usage.input_tokens / 1e6 * OPUS_INPUT_PER_M
    out_cost = usage.output_tokens / 1e6 * OPUS_OUTPUT_PER_M
    return in_cost + out_cost

def call_opus(prompt: str, max_tokens: int = 8000, temperature: float = None) -> Dict[str, Any]:
    """Call Opus with retries. Returns dict with keys: text, parsed_json (or None), usage, cost.

    Note: Opus 4.7 deprecated the temperature parameter (model sets its own sampling
    internally). The `temperature` argument is accepted for backwards compatibility
    with older callers but is ignored.
    """
    global _cum_input_tokens, _cum_output_tokens, _cum_cost

    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            # Opus 4.7 deprecated `temperature` — do not pass it.
            response = client.messages.create(
                model=MODEL_ID,
                max_tokens=max_tokens,
                messages=[{'role': 'user', 'content': prompt}],
            )

            text = response.content[0].text if response.content else ''
            cost = estimate_cost(response.usage)

            _cum_input_tokens += response.usage.input_tokens
            _cum_output_tokens += response.usage.output_tokens
            _cum_cost += cost

            if _cum_cost > COST_CEILING_USD:
                raise RuntimeError(
                    f'Cost ceiling ${COST_CEILING_USD:.2f} exceeded (${_cum_cost:.4f}). Aborting.'
                )

            # Parse JSON. Be tolerant of leading/trailing markdown fences.
            parsed = None
            cleaned = text.strip()
            if cleaned.startswith('```'):
                # Strip ```json ... ``` or ``` ... ```
                lines = cleaned.split('\n')
                if lines[0].startswith('```'):
                    lines = lines[1:]
                if lines and lines[-1].strip() == '```':
                    lines = lines[:-1]
                cleaned = '\n'.join(lines)
            try:
                parsed = json.loads(cleaned)
            except json.JSONDecodeError as je:
                print(f'  [JSON parse failed on attempt {attempt+1}/{MAX_RETRIES}] {je}')
                # Save the bad output for debugging
                debug_path = RAW_DIR / f'_failed_parse_attempt{attempt+1}_{int(time.time())}.txt'
                debug_path.write_text(text)
                if attempt < MAX_RETRIES - 1:
                    time.sleep(RETRY_BASE_DELAY ** (attempt + 1))
                    continue

            return {
                'text': text,
                'parsed_json': parsed,
                'usage': {
                    'input_tokens': response.usage.input_tokens,
                    'output_tokens': response.usage.output_tokens,
                },
                'cost': cost,
            }
        except Exception as e:
            last_err = e
            wait = RETRY_BASE_DELAY ** (attempt + 1)
            print(f'  [API error attempt {attempt+1}/{MAX_RETRIES}] {type(e).__name__}: {e}; waiting {wait}s')
            if attempt < MAX_RETRIES - 1:
                time.sleep(wait)

    raise RuntimeError(f'Opus call failed after {MAX_RETRIES} retries: {last_err}')

print('Opus caller ready (temperature parameter is accepted for compatibility but ignored).')


## 9. Format response blocks for prompts

In [ ]:
def format_responses_block(sample_df: pd.DataFrame) -> str:
    """Format a sampled responses dataframe into a numbered text block for prompts.

    Shuffles to break any ordering by condition. Numbers responses R1, R2, ... so
    Opus can reference them in anchors.
    """
    rng = np.random.RandomState(RANDOM_SEED)
    shuffled = sample_df.sample(frac=1, random_state=rng).reset_index(drop=True)

    lines = []
    for i, row in shuffled.iterrows():
        text = str(row['response']).strip().replace('\n', ' ')
        lines.append(f'R{i+1}: {text}')
    return '\n\n'.join(lines)

# Spot check on the first question
example_block = format_responses_block(samples_by_question[TARGET_QUESTIONS[0]])
print(f'Example response block for {TARGET_QUESTIONS[0]} (first 1500 chars):')
print('-' * 80)
print(example_block[:1500])
print('...' if len(example_block) > 1500 else '')

## 10. Run Pass 1 (inductive)

One Opus call per question. Each call sees only the responses for that question, no condition labels, no priming on what themes to look for.

In [ ]:
def build_scenario_block(question_id: str) -> str:
    """Construct a scenario-context block for the prompt, or empty string if no context."""
    sc = SCENARIO_CONTEXT.get(question_id)
    if not sc:
        return ''
    return (
        'Before answering, participants read this scenario:\n'
        f'"""{sc}"""\n\n'
        'The question below directly references this scenario. Keep this context in '
        'mind when interpreting responses.\n\n'
    )

pass1_results = {}

for q in TARGET_QUESTIONS:
    print(f'\n=== Pass 1: {q} ===')
    sample_df = samples_by_question[q]
    responses_block = format_responses_block(sample_df)

    prompt = PASS1_PROMPT_TEMPLATE.format(
        scenario_block=build_scenario_block(q),
        question_text=QUESTION_TEXTS[q],
        question_id=q,
        n_responses=len(sample_df),
        responses_block=responses_block,
    )

    result = call_opus(prompt, max_tokens=6000)
    pass1_results[q] = result

    # Save raw text for transparency
    (RAW_DIR / f'pass1_{q}.txt').write_text(result['text'])
    (RAW_DIR / f'pass1_{q}_prompt.txt').write_text(prompt)

    if result['parsed_json'] is None:
        print(f'  [WARN] JSON parse failed for {q}, raw output saved to {RAW_DIR}')
    else:
        n_themes = len(result['parsed_json'].get('themes', []))
        print(f'  [OK] {n_themes} themes identified, '
              f'tokens=({result["usage"]["input_tokens"]}, {result["usage"]["output_tokens"]}), '
              f'cost=${result["cost"]:.4f}')

print(f'\nPass 1 complete. Cumulative cost so far: ${_cum_cost:.4f}')


### 10.1 Quick look at Pass 1 themes per question

In [ ]:
for q in TARGET_QUESTIONS:
    parsed = pass1_results[q].get('parsed_json')
    if parsed is None:
        print(f'\n=== {q} === [PARSE FAILED]')
        continue
    print(f'\n=== {q} ===')
    for t in parsed.get('themes', []):
        cat = t.get('category', '—') or '—'
        prev = t.get('prevalence_estimate', '?')
        print(f'  [{cat}] {t.get("code_id", "?"):<32s} ({prev:>12s})  {t.get("label", "")}')

## 11. Run Pass 2 (refinement)

For each question, send Opus (a) the original responses, (b) its own Pass 1 themes, and ask for merging / splitting / category grouping / definition tightening.

In [ ]:
pass2_results = {}

for q in TARGET_QUESTIONS:
    print(f'\n=== Pass 2: {q} ===')

    pass1_parsed = pass1_results[q].get('parsed_json')
    if pass1_parsed is None:
        print(f'  [SKIP] No Pass 1 output to refine')
        pass2_results[q] = {'parsed_json': None, 'text': '', 'cost': 0.0}
        continue

    sample_df = samples_by_question[q]
    responses_block = format_responses_block(sample_df)

    prompt = PASS2_PROMPT_TEMPLATE.format(
        scenario_block=build_scenario_block(q),
        question_text=QUESTION_TEXTS[q],
        question_id=q,
        responses_block=responses_block,
        pass1_json=json.dumps(pass1_parsed, indent=2),
    )

    result = call_opus(prompt, max_tokens=8000)
    pass2_results[q] = result

    (RAW_DIR / f'pass2_{q}.txt').write_text(result['text'])
    (RAW_DIR / f'pass2_{q}_prompt.txt').write_text(prompt)

    if result['parsed_json'] is None:
        print(f'  [WARN] JSON parse failed for {q}, raw output saved to {RAW_DIR}')
    else:
        n_themes = len(result['parsed_json'].get('themes', []))
        n_cats = len(result['parsed_json'].get('categories', {}))
        print(f'  [OK] {n_themes} themes in {n_cats} categories, '
              f'tokens=({result["usage"]["input_tokens"]}, {result["usage"]["output_tokens"]}), '
              f'cost=${result["cost"]:.4f}')

print(f'\nPass 2 complete. Total cost: ${_cum_cost:.4f}')


## 12. Assemble final rubric

Combines Pass 2 outputs into a single `rubric_v2.json` for Notebook 04.


In [ ]:
rubric = {
    'version': '2.0',
    'study': 'Study 2 — Chatbot Neuroticism',
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'rubric_developed_by': MODEL_ID,
    'rubric_development_method': 'two-pass inductive open coding (Pass 1: discovery; Pass 2: refinement)',
    'methodological_disclosure': (
        'Rubric was developed by a single LLM (Claude Opus 4.7) on a stratified sample of '
        f'{N_PER_CONDITION_PER_QUESTION * 2} responses per question '
        f'({N_PER_CONDITION_PER_QUESTION} LowN + {N_PER_CONDITION_PER_QUESTION} HighN), '
        '96 responses total. Opus 4.7\'s biases are baked into the rubric. Notebook 04 '
        'uses three LLMs from three families (Claude Opus 4.6, GPT-5.4-mini, Gemini 3 Flash) '
        'to code all OE responses against this locked rubric, providing cross-family coding '
        'reliability even though rubric development was within-family.'
    ),
    'sample': {
        'random_seed': RANDOM_SEED,
        'n_per_condition_per_question': N_PER_CONDITION_PER_QUESTION,
        'min_words': MIN_WORDS,
        'sample_ids_path': str((RUBRIC_DIR / 'sample_ids.json').relative_to(PROJECT_ROOT)),
    },
    'questions': {},
    'cost_to_develop_usd': round(_cum_cost, 4),
    'tokens_used': {
        'input': _cum_input_tokens,
        'output': _cum_output_tokens,
    },
}

# Total non-blank counts in the full corpus, per question (helpful for Notebook 04)
total_in_data = nonblank.groupby('oe_field').size().to_dict()

for q in TARGET_QUESTIONS:
    p2 = pass2_results[q].get('parsed_json')
    if p2 is None:
        # Fall back to pass 1 if pass 2 failed
        p1 = pass1_results[q].get('parsed_json')
        if p1 is not None:
            print(f'[NOTE] {q}: using Pass 1 themes (Pass 2 parse failed)')
            rubric['questions'][q] = {
                'question_id': q,
                'question_text': QUESTION_TEXTS[q],
                'n_in_data': int(total_in_data.get(q, 0)),
                'n_in_open_coding_sample': len(samples_by_question[q]),
                'pass2_status': 'failed_parse',
                'categories': {},
                'themes': p1.get('themes', []),
                'rubric_notes': '',
            }
        else:
            print(f'[ERROR] {q}: both passes failed parse, no rubric available')
            continue
    else:
        rubric['questions'][q] = {
            'question_id': q,
            'question_text': QUESTION_TEXTS[q],
            'n_in_data': int(total_in_data.get(q, 0)),
            'n_in_open_coding_sample': len(samples_by_question[q]),
            'pass2_status': 'ok',
            'categories': p2.get('categories', {}),
            'themes': p2.get('themes', []),
            'rubric_notes': p2.get('rubric_notes', ''),
        }

    n_themes = len(rubric['questions'][q]['themes'])
    n_cats = len(rubric['questions'][q]['categories'])
    print(f'  [OK] {q}: {n_themes} themes in {n_cats} categories')

# Save JSON — OVERWRITES rubric_v2.json (the prior Opus-developed v2)
rubric_path = RUBRIC_DIR / 'rubric_v2.json'
with open(rubric_path, 'w') as f:
    json.dump(rubric, f, indent=2, ensure_ascii=False)
print(f'\nWrote {rubric_path} ({rubric_path.stat().st_size / 1024:.1f} KB)')
print(f'(This overwrites the prior Opus 4.6 v2; the original v1 is preserved if present.)')


### 12.1 Generate human-readable Markdown version

For David and Alice to read on their phones if needed.

In [ ]:
def render_rubric_markdown(rubric: Dict[str, Any]) -> str:
    md_lines = []
    md_lines.append(f'# Rubric v{rubric["version"]} — {rubric["study"]}')
    md_lines.append('')
    md_lines.append(f'**Created:** {rubric["created_at"]}  ')
    md_lines.append(f'**Developed by:** {rubric["rubric_developed_by"]}  ')
    md_lines.append(f'**Method:** {rubric["rubric_development_method"]}  ')
    md_lines.append(f'**Cost to develop:** \\${rubric["cost_to_develop_usd"]:.4f}  ')
    md_lines.append('')
    md_lines.append('## Methodological disclosure')
    md_lines.append('')
    md_lines.append(rubric['methodological_disclosure'])
    md_lines.append('')

    for q, qdata in rubric['questions'].items():
        md_lines.append('---')
        md_lines.append('')
        md_lines.append(f'## {q}')
        md_lines.append('')
        md_lines.append(f'**Question text:** {qdata["question_text"]}')
        md_lines.append('')
        md_lines.append(f'**Sample size:** {qdata["n_in_open_coding_sample"]} responses sampled '
                        f'from {qdata["n_in_data"]} non-blank responses in data')
        md_lines.append('')
        if qdata.get('rubric_notes'):
            md_lines.append(f'**Opus\'s notes on this rubric:** {qdata["rubric_notes"]}')
            md_lines.append('')

        # Categories
        if qdata.get('categories'):
            md_lines.append('### Categories')
            md_lines.append('')
            for cat_id, cat_info in qdata['categories'].items():
                md_lines.append(f'- **`{cat_id}`** ({cat_info.get("coding_type", "?")}) — '
                                f'{cat_info.get("label", "")}: '
                                f'{cat_info.get("description", "")}')
            md_lines.append('')

        # Themes
        md_lines.append('### Themes')
        md_lines.append('')
        for t in qdata.get('themes', []):
            md_lines.append(f'#### `{t.get("code_id", "?")}` — {t.get("label", "")}')
            md_lines.append('')
            md_lines.append(f'- **Category:** `{t.get("category", "—")}`')
            md_lines.append(f'- **Coding type:** {t.get("coding_type", "?")}')
            md_lines.append(f'- **Prevalence:** {t.get("prevalence_estimate", "?")}')
            md_lines.append(f'- **Definition:** {t.get("definition", "")}')

            anchors = t.get('positive_anchors', [])
            if anchors:
                md_lines.append('- **Positive anchors:**')
                for a in anchors:
                    md_lines.append(f'    - {a}')

            negs = t.get('negative_anchors', [])
            if negs:
                md_lines.append('- **Negative anchors:**')
                for a in negs:
                    md_lines.append(f'    - {a}')

            if t.get('changes_from_pass1'):
                md_lines.append(f'- **Pass 2 change:** {t["changes_from_pass1"]}')
            md_lines.append('')

    return '\n'.join(md_lines)

md_text = render_rubric_markdown(rubric)
md_path = RUBRIC_DIR / 'rubric_v2.md'
md_path.write_text(md_text)
print(f'Wrote {md_path} ({md_path.stat().st_size / 1024:.1f} KB)')


## 13. Review summary

The next two cells print the rubric structure and a per-theme summary so you can do a fast first-pass review without opening the file. The full review for editing happens against `rubric_v2.md` and `rubric_v2.json` directly.


In [ ]:
print('=' * 80)
print('RUBRIC v2 STRUCTURE (Opus 4.7 development)')
print('=' * 80)
print(f'Total cost: ${_cum_cost:.4f}')
print(f'Total tokens: {_cum_input_tokens:,} in / {_cum_output_tokens:,} out')
print()

for q, qdata in rubric['questions'].items():
    print(f'\n{q}')
    print(f'  Question text: {qdata["question_text"][:90]}...'
          if len(qdata['question_text']) > 90 else f'  Question text: {qdata["question_text"]}')
    print(f'  Sample: {qdata["n_in_open_coding_sample"]} of {qdata["n_in_data"]} responses')
    print(f'  Categories ({len(qdata.get("categories", {}))}):')
    for cat_id, cat_info in qdata.get('categories', {}).items():
        n_in_cat = sum(1 for t in qdata.get('themes', []) if t.get('category') == cat_id)
        print(f'    - {cat_id} ({cat_info.get("coding_type", "?")}, {n_in_cat} themes): '
              f'{cat_info.get("label", "")}')
    print(f'  Themes ({len(qdata.get("themes", []))}):')
    for t in qdata.get('themes', []):
        print(f'    - [{t.get("category", "?"):<20s}] {t.get("code_id", "?"):<32s}  '
              f'({t.get("prevalence_estimate", "?")})')


In [ ]:
# Print the full rubric markdown to the cell output for fast review
print(md_text)

## 14. Wrap-up checklist

Before running Notebook 04, you and Alice should:

- [ ] Open `rubric_v2.md` and read every theme. Anything obviously wrong, missing, or duplicated?
- [ ] Pay particular attention to `oe_responsibility` and `oe_compensation` — these drive the manuscript headline.
- [ ] If you want to edit the rubric, edit `rubric_v2.json` directly (it's the file Notebook 04 reads). Save edits to the same filename — there is no need to bump the version unless the changes are substantive enough that comparing to the model-developed version is informative.
- [ ] If you remove a theme, also remove its anchors. If you add a theme, write the definition and add at least 2 anchors drawn verbatim from the data.
- [ ] Confirm `data/derived/rubric/sample_ids.json` exists (so the sample is reproducible).
- [ ] Do not re-run this notebook with the same seed unless you want the same sample. The seed is locked at 42 for reproducibility.

If anything looks off (parse failures, missing themes, suspicious anchors), check `data/derived/rubric/_raw/` for the raw model outputs.


In [ ]:
print('=' * 70)
print('NOTEBOOK 03 COMPLETE')
print('=' * 70)
print(f'Rubric:      {rubric_path}')
print(f'Markdown:    {md_path}')
print(f'Sample IDs:  {RUBRIC_DIR / "sample_ids.json"}')
print(f'Raw outputs: {RAW_DIR}')
print()
print(f'Total spend this run: ${_cum_cost:.4f}')
print(f'Total tokens: {_cum_input_tokens:,} in / {_cum_output_tokens:,} out')
print()
print('Next: review rubric_v2.md. Edit rubric_v2.json in place if changes are needed.')
print('Then run Notebook 04 (LLM coding of all OE responses with Opus 4.6 + GPT-5.4-mini + Gemini 3 Flash).')
